In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_INPUTS = True
REUSE_METRICS = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Coronary Anatomy Reconciliation v1.1

Uses only explicitly selected prior artifacts: validated RCA, independently frozen LAD, prior trunk candidate, and prior secondary-branch candidate. Every centerline is converted through source-series-7 DICOM geometry into LPS millimetres before comparison. If a prior cache lacks full DICOM geometry, v1.1 reconstructs it directly from `Full_DICOM.zip`. No vessel relabeling is performed automatically.


In [ ]:
!pip -q install scipy pandas matplotlib SimpleITK pydicom
import os, shutil, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -q --depth 1 --branch coronary-anatomy-reconciliation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.coronary_anatomy_reconciliation_v2 import CoronaryAnatomyReconciliationWorkflowV2, synthetic_reconciliation_self_test
test=synthetic_reconciliation_self_test(); display(test); assert test['passed'], test
wf=CoronaryAnatomyReconciliationWorkflowV2(root='/content/drive/MyDrive/OpenPlaque', reuse={'inputs':REUSE_INPUTS,'metrics':REUSE_METRICS,'figures':REUSE_FIGURES,'report':REUSE_REPORT})
display(wf.cache_status())


In [ ]:
snapshot=wf.load_inputs(); display(snapshot)
print('Resolved inputs:')
for k,v in wf.source_files.items(): print(f'  {k}: {v}')


In [ ]:
summary=wf.compute_metrics(); display(summary)
print('Pairwise common-frame distances:'); display(wf.pairwise.sort_values('minimum_distance_mm'))
print('Closest endpoint relationships:'); display(wf.endpoints.sort_values('distance_mm').head(20))


In [ ]:
qc,_=wf.build_junction_qc(); print('Nearest-approach source-CCTA QC:'); display(qc)


In [ ]:
names=wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report=wf.build_report(); zip_path=wf.package()
print('STATUS:', wf.summary['status'])
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_CORONARY_ANATOMY_RECONCILIATION_REPORT_BACK.zip')
